# 01_market_prices_autoloader
- **Purpose**: This notebook is reserved for loading market price CSV files into the Bronze layer.
- **Source domain**: `sample_data/market_prices/`
- **Expected Bronze table**: `vattenfall_dev.raw.bronze_market_prices`

**Expected fields**
- event_date
- region
- market_type
- price_eur_mwh
- volume_mwh
- source_system
- last_updated_ts

In [0]:
from pyspark.sql import functions as F

catalog = "vattenfall_dev"
schema = "raw"

source_domain = "market_prices"

landing_path = f"/Volumes/{catalog}/{schema}/landing/market_prices"
checkpoint_path = f"/Volumes/{catalog}/{schema}/checkpoints/market_prices_checkpoint"
schema_path = f"/Volumes/{catalog}/{schema}/checkpoints/market_prices_schema"
bronze_table = f"{catalog}.{schema}.bronze_market_prices"

print("Source domain:", source_domain)
print("Landing path:", landing_path)
print("Checkpoint path:", checkpoint_path)
print("Schema path:", schema_path)
print("Bronze target table:", bronze_table)

display(dbutils.fs.ls(landing_path))

market_prices_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(landing_path)
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

query = (
    market_prices_stream_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

bronze_df = spark.table(bronze_table)

print("Rows in bronze after ingestion:", bronze_df.count())
print("Columns in bronze:", bronze_df.columns)
display(bronze_df.limit(20))